# EGNN CP3D

In [1]:
from src.dataset import SE3TransformerCP3DDataModule, EGNNQM9DataModule
from egnn.qm9 import dataset
import numpy as np
import torch

/home/pham/miniforge3/envs/cp3d/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with np.load("pdb_data/qm9/test.npz") as f:
    ds = {key: torch.from_numpy(val) for key, val in f.items()}

In [3]:
print(torch.min(ds["num_atoms"]))
print(torch.max(ds["num_atoms"]))
ds["charges"].shape

tensor(7)
tensor(29)


torch.Size([13083, 29])

In [4]:
ds["charges"].shape

torch.Size([13083, 29])

In [5]:
dm = EGNNQM9DataModule("pdb_data/qm9")

In [6]:
dm.ds_val.data["charges"].shape

torch.Size([17748, 29])

In [7]:
dm.ds_val.data["homo"][0]

tensor(-0.2928)

In [8]:
datamodule = SE3TransformerCP3DDataModule("pdb_data/Hexene", seed=43)

In [9]:
datamodule.ds_test.dataset[0].y.squeeze()

tensor(0.4050)

In [10]:
dataloaders, charge_scale = dataset.retrieve_dataloaders(4, 2)

In [11]:
batch = next(iter(dataloaders["train"]))

In [12]:
batch["charges"].shape

torch.Size([4, 21])

In [13]:
batch["positions"][1]

tensor([[-5.8959e-02,  1.2280e+00,  7.3408e-03],
        [-2.4918e-02, -2.9284e-02,  7.3784e-04],
        [-1.2632e+00, -9.5175e-01,  2.5879e-03],
        [-5.8163e-01, -2.2755e+00, -7.8026e-03],
        [ 7.6898e-01, -2.1077e+00, -1.4124e-02],
        [ 1.1492e+00, -7.7686e-01, -9.3240e-03],
        [ 1.4166e+00, -3.3606e+00, -2.3997e-02],
        [ 3.8332e-01, -4.2851e+00, -2.3360e-02],
        [-8.2574e-01, -3.6232e+00, -1.3418e-02],
        [ 8.8546e-01,  1.6163e+00,  4.3120e-03],
        [-1.8739e+00, -7.4195e-01,  8.8833e-01],
        [-1.8833e+00, -7.3294e-01, -8.7443e-01],
        [ 2.4711e+00, -3.5807e+00, -3.0722e-02],
        [ 4.1856e-01, -5.3622e+00, -2.9146e-02],
        [-1.7252e+00, -4.0702e+00, -1.0885e-02],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00

In [14]:
datamodule = SE3TransformerCP3DDataModule("pdb_data/Hexene", 1, 1, split=0)

In [15]:
point = datamodule.ds_train[0]

In [16]:
point

Data(x=[112, 11], edge_index=[2, 232], edge_attr=[232, 4], y=[1, 1], pos=[112, 3], z=[112], smiles='[H]c1c([H])c([H])c(C([H])([H])[C@@]2([H])C(=O)N(C([H])([H])C([H])([H])C([H])([H])[H])C([H])([H])C(O)N([H])[C@]([H])(C([H])([H])C([H])(C([H])([H])[H])C([H])([H])[H])C(=O)N([H])C([H])(C([H])([H])C3C([H])C([H])C([H])C([H])C3[H])C([H])([H])C(O)N3C([H])([H])C([H])([H])C([H])([H])[C@]3([H])C(=O)N(C([H])([H])C3C([H])C([H])C([H])C([H])C3[H])C([H])([H])C(O)N2[H])c([H])c1[H]', name='2020_Townsend_2765', idx=[1])

In [17]:
point.x[:5]

tensor([[0., 0., 1., 0., 0., 7., 0., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0., 6., 0., 0., 0., 1., 2.],
        [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 6., 0., 0., 0., 1., 2.]])

# CPMP QM9

In [1]:
import pandas as pd
import numpy as np
import torch

from tqdm import tqdm
from rdkit import Chem, RDLogger
from joblib import Parallel, delayed
from cpmp.featurization.data_utils import featurize_mol, Molecule, MolDataset

RDLogger.DisableLog("rdApp.*")

In [2]:
# import pathlib
# data_dir = pathlib.Path("owncodes/DataProcessor/pkl")
# X_train = pd.read_pickle(data_dir / "X_train_mmff_False_0.pkl")
# y_train = pd.read_pickle(data_dir / "y_train_mmff_False_0.pkl")

In [3]:
suppl = Chem.SDMolSupplier("pdb_data/cpmp/qm9/gdb9.sdf", removeHs=False, sanitize=False)

In [4]:
HAR2EV = 27.211386246
KCALMOL2EV = 0.04336414
# fmt: off
conversion = np.array(
    [1., 1., HAR2EV, HAR2EV, HAR2EV, 1., HAR2EV, HAR2EV, HAR2EV, HAR2EV, HAR2EV,
    1., KCALMOL2EV, KCALMOL2EV, KCALMOL2EV, KCALMOL2EV, 1., 1., 1.]
)
# fmt: on

In [5]:
y_df = pd.read_csv(
    "pdb_data/cpmp/qm9/gdb9.sdf.csv",
    usecols=range(1, 20),  # select columns by position (exclude col 0)
    dtype=np.float32,
)

# Move first 3 columns to the end
y_df = y_df.iloc[:, list(range(3, y_df.shape[1])) + [0, 1, 2]]

# Apply unit conversion
y_df = y_df.mul(conversion.astype(np.float32), axis=1)

In [6]:
with Parallel(n_jobs=-1) as pool:
    x_all = pool(
        delayed(featurize_mol)(mol, True, False)
        for mol in tqdm(suppl, total=len(suppl))
    )

100%|██████████| 133885/133885 [00:09<00:00, 13436.37it/s]


In [7]:
# x_all = []

# for i, mol in tqdm(enumerate(suppl), total=len(suppl)):
#     afm, adj, dist = featurize_mol(mol, True, False)
#     x_all.append([afm, adj, dist])

In [8]:
def construct_dataset(x_all, y_df):
    """Construct a MolDataset object from the provided data.

    Args:
        x_all (list): A list of molecule features.
        y_all (list): A list of the corresponding labels.

    Returns:
        A MolDataset object filled with the provided data.
    """
    output = [
        Molecule(data[0], data[1], i)
        for i, data in enumerate(zip(x_all, y_df.to_dict(orient="records")))
    ]
    return MolDataset(output)

In [9]:
ds = construct_dataset(x_all, y_df)

In [10]:
def _get_split_sizes(full_dataset) -> tuple[int, int, int]:
    len_full = len(full_dataset)
    len_train = 100_000
    len_test = int(0.1 * len_full)
    len_val = len_full - len_train - len_test
    return len_train, len_val, len_test


ds_train, ds_val, ds_test = torch.utils.data.random_split(
    ds,
    _get_split_sizes(ds),
    generator=torch.Generator().manual_seed(1),
)

In [11]:
ds_train.dataset.data_list[0].node_features.shape[1]

26

In [13]:
ds_train.dataset.data_list[0].y

{'mu': 0.0,
 'alpha': 13.210000038146973,
 'homo': -10.549854278564453,
 'lumo': 3.186453342437744,
 'gap': 13.736308097839355,
 'r2': 35.36410140991211,
 'zpve': 1.2176822423934937,
 'u0': -1101.48779296875,
 'u298': -1101.4097900390625,
 'h298': -1101.384033203125,
 'g298': -1102.02294921875,
 'cv': 6.468999862670898,
 'u0_atom': -17.172182083129883,
 'u298_atom': -17.286823272705078,
 'h298_atom': -17.38965606689453,
 'g298_atom': -16.151918411254883,
 'A': 157.71180725097656,
 'B': 157.70997619628906,
 'C': 157.7069854736328}

# EGNN QM9

In [1]:
from egnn.qm9.data.prepare import prepare_dataset
from egnn.qm9.data.utils import get_species
from egnn.qm9.data.dataset import ProcessedDataset
import numpy as np
import torch

In [2]:
datafiles = prepare_dataset("data/egnn/QM9", "qm9")

In [3]:
datafiles

{'train': 'data/egnn/QM9/qm9/train.npz',
 'valid': 'data/egnn/QM9/qm9/valid.npz',
 'test': 'data/egnn/QM9/qm9/test.npz'}

In [4]:
datasets = {}
for split, datafile in datafiles.items():
    with np.load(datafile) as f:
        datasets[split] = {key: torch.from_numpy(val) for key, val in f.items()}
keys = [list(data.keys()) for data in datasets.values()]

all_species = get_species(datasets, ignore_check=False)

datasets = {
    split: ProcessedDataset(
        data,
        included_species=all_species,
        subtract_thermo=True,
    )
    for split, data in datasets.items()
}

In [5]:
datasets

{'train': <egnn.qm9.data.dataset.ProcessedDataset at 0x7ee90aa5cd50>,
 'valid': <egnn.qm9.data.dataset.ProcessedDataset at 0x7ee90946a610>,
 'test': <egnn.qm9.data.dataset.ProcessedDataset at 0x7ee90945cb90>}

In [7]:
print(len(datasets["train"]))
print(len(datasets["valid"]))
print(len(datasets["test"]))

100000
17748
13083


In [1]:
from se3_transformer.se3_transformer.data_loading.cycpept3D import CycPept3DDataset
import pathlib
import torch
from egnn.qm9.data.dataset import ProcessedDataset
from torch.utils.data import Subset, DataLoader
import pandas as pd

/home/pham/miniforge3/envs/cp3d/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_dir = pathlib.Path("data/egnn/Hexene")

In [3]:
raw_dataset = CycPept3DDataset(root=data_dir)

In [4]:
csv_path = data_dir / "raw/Random_Split_With_PDB.csv"

In [5]:
split = 9
split_col = f"split{split}"
split_df = pd.read_csv(
    csv_path,
    usecols=["CycPeptMPDB_ID", "Source", split_col],
    dtype={"CycPeptMPDB_ID": str, "Source": str},
).assign(
    Source_ID=lambda x: x["Source"].str.cat(x["CycPeptMPDB_ID"].astype(str), sep="_")
)
split_dict = split_df.set_index("Source_ID")[split_col].to_dict()

train_indices = []
val_indices = []
test_indices = []
dispatch = {
    "train": train_indices.append,
    "valid": val_indices.append,
    "test": test_indices.append,
}

for i, data in enumerate(raw_dataset):
    name = data.name
    try:
        split_type = split_dict[name]
    except KeyError:
        raise ValueError(f"Data point '{name}' not found in split dictionary.")

    try:
        func = dispatch[split_type]
    except KeyError:
        raise ValueError(
            f"Unknown split type '{split_type}' for {name}. Expected 'train', 'valid', or 'test'."
        )
    func(i)

ds_train = Subset(raw_dataset, train_indices)
ds_val = Subset(raw_dataset, val_indices)
ds_test = Subset(raw_dataset, test_indices)

In [8]:
data = raw_dataset[0]

In [11]:
def to_processed_dataset(ds):
    raw_props = {
        "charges": [],
        "positions": [],
        "num_atoms": [],
        "one_hot": [],
        "pampa": [],
        "name": [],
    }
    for data in ds:
        raw_props["charges"].append(data.z)
        raw_props["positions"].append(data.pos)
        raw_props["num_atoms"].append(data.z.shape[0])
        raw_props["one_hot"].append(data.x[:, :5] == 1.0)
        raw_props["pampa"].append(data.y.squeeze())
        raw_props["name"].append(data.name)
    raw_props_padded = {}
    raw_props_padded["charges"] = torch.nn.utils.rnn.pad_sequence(
        raw_props["charges"], batch_first=True, padding_value=0
    )
    raw_props_padded["positions"] = torch.nn.utils.rnn.pad_sequence(
        raw_props["positions"], batch_first=True, padding_value=0.0
    )
    raw_props_padded["one_hot"] = torch.nn.utils.rnn.pad_sequence(
        raw_props["one_hot"], batch_first=True, padding_value=False
    )
    raw_props_padded["num_atoms"] = torch.tensor(raw_props["num_atoms"])
    raw_props_padded["pampa"] = torch.tensor(raw_props["pampa"])
    raw_props_padded["name"] = raw_props["name"]

    processed_ds = ProcessedDataset(data=raw_props_padded, subtract_thermo=False)

    return processed_ds

In [12]:
train_ds = to_processed_dataset(ds_train)
val_ds = to_processed_dataset(ds_val)
test_ds = to_processed_dataset(ds_test)

In [13]:
train_ds